# ABPT Qwen Motif Colab Runner

Этот ноутбук поднимает окружение в **Google Colab** и прогоняет текущий `Qwen motif` стек:

1. Клонирует репозиторий
2. Устанавливает зависимости
3. Гоняет целевые `pytest`-тесты для `qwen_motif` модулей
4. Запускает staged smoke experiments через `scripts/run_qwen_motif_stage_experiments.py`
5. Опционально делает `pretrained smoke` на `Qwen/Qwen2.5-1.5B`

Рекомендуемый runtime: **GPU (T4/A100/L4)**, но базовый smoke run сработает и на CPU.

Открыть в Colab после пуша в GitHub можно по ссылке:  
`https://colab.research.google.com/github/kharkilirov1/Anchor-engine/blob/main/ABPT_Qwen_Motif_Colab.ipynb`

## 0. Проверка runtime

In [ ]:
import os
import platform
import subprocess
import sys

import torch

print(f'Python: {sys.version}')
print(f'Platform: {platform.platform()}')
print(f'Torch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    print(f'VRAM: {props.total_memory / 1e9:.2f} GB')
else:
    print('GPU не обнаружен. Для нормального Colab прогона лучше переключить Runtime -> GPU.')

try:
    print(subprocess.check_output(['nvidia-smi'], text=True))
except Exception as exc:
    print(f'nvidia-smi unavailable: {exc}')

## 1. Клонирование репозитория

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/kharkilirov1/Anchor-engine.git'
REPO_DIR = Path('/content/ABPT')
os.environ.setdefault('HF_HOME', '/content/.cache/huggingface')

if REPO_DIR.exists():
    print('Репозиторий уже существует -> git pull')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print(f'cwd = {Path.cwd()}')
print(subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())
print(subprocess.check_output(['git', 'status', '--short'], text=True))

## 2. Установка зависимостей

In [ ]:
%%bash
set -euo pipefail
python -m pip install -q --upgrade pip
python -m pip install -q -r requirements.txt accelerate psutil
python -m pip install -q "transformers>=4.46.0"
python -m pip install -q pytest
echo 'Dependencies installed'

## 3. Sanity check импортов

In [ ]:
import importlib

modules = [
    'torch',
    'transformers',
    'einops',
    'pytest',
]
for name in modules:
    module = importlib.import_module(name)
    print(name, getattr(module, '__version__', 'n/a'))

## 4. Целевые тесты для Qwen motif модулей

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, '-m', 'pytest',
    'tests/test_qwen_motif_ffn.py',
    'tests/test_qwen_motif_patch.py',
    'tests/test_qwen_motif_lora.py',
    'tests/test_qwen_motif_attention.py',
    '-q',
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

## 5. Параметры staged smoke run

При GPU можно поднять число шагов. На CPU лучше оставить короткий профиль.

In [ ]:
from datetime import datetime

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RUN_TAG = datetime.utcnow().strftime('%Y%m%d_%H%M%S')

if DEVICE == 'cuda':
    PROFILE = {
        'router_steps': 16,
        'lora_steps': 16,
        'attention_steps': 16,
        'sparse_steps': 8,
        'batch_size': 8,
    }
else:
    PROFILE = {
        'router_steps': 8,
        'lora_steps': 8,
        'attention_steps': 8,
        'sparse_steps': 6,
        'batch_size': 4,
    }

OUTPUT_JSON = f'archive/qwen_motif_stage_experiments_colab_{DEVICE}_{RUN_TAG}.json'
PROFILE, OUTPUT_JSON

## 6. Запуск staged experiments

In [ ]:
import shlex
import subprocess
import sys

cmd = [
    sys.executable,
    'scripts/run_qwen_motif_stage_experiments.py',
    '--device', DEVICE,
    '--router_steps', str(PROFILE['router_steps']),
    '--lora_steps', str(PROFILE['lora_steps']),
    '--attention_steps', str(PROFILE['attention_steps']),
    '--sparse_steps', str(PROFILE['sparse_steps']),
    '--batch_size', str(PROFILE['batch_size']),
    '--output_json', OUTPUT_JSON,
]
print('Running command:')
print(' '.join(shlex.quote(part) for part in cmd))
subprocess.run(cmd, check=True)
print(f'Saved to {OUTPUT_JSON}')

## 7. Краткая сводка по результатам

In [ ]:
import json
from pathlib import Path

payload = json.loads(Path(OUTPUT_JSON).read_text(encoding='utf-8'))

for stage_name in ['router_only', 'ffn_lora', 'attention_soft', 'attention_sparse']:
    stage = payload[stage_name]
    history = stage.get('history', [])
    last_loss = history[-1]['loss'] if history else None
    print(f'{stage_name}: trainable={stage["trainable_parameters"]}, last_loss={last_loss}')

print('\nDomain stats from attention_sparse:')
for domain, stats in payload['attention_sparse']['eval'].items():
    print(f'- {domain}: loss={stats["loss"]:.4f}')
    if stats['ffn_mean_alpha']:
        print('  ffn_mean_alpha:', stats['ffn_mean_alpha'])
    if stats['attn_mean_alpha']:
        print('  attn_mean_alpha:', stats['attn_mean_alpha'])

## 8. Опционально: pretrained smoke на Qwen2.5-1.5B

По умолчанию ячейка ничего не запускает. Поставь `RUN_PRETRAINED_SMOKE = True`, если хочешь проверить патчинг на реальном open-weight checkpoint.

In [ ]:
import shlex
import subprocess
import sys

RUN_PRETRAINED_SMOKE = False
PRETRAINED_MODEL = 'Qwen/Qwen2.5-1.5B'
PRETRAINED_OUTPUT_JSON = f'archive/qwen_motif_pretrained_smoke_{DEVICE}_{RUN_TAG}.json'

if RUN_PRETRAINED_SMOKE:
    cmd = [
        sys.executable,
        'scripts/run_qwen_motif_stage_experiments.py',
        '--device', DEVICE,
        '--router_steps', '2',
        '--lora_steps', '2',
        '--attention_steps', '2',
        '--sparse_steps', '2',
        '--batch_size', '2',
        '--run_pretrained_smoke',
        '--pretrained_model', PRETRAINED_MODEL,
        '--output_json', PRETRAINED_OUTPUT_JSON,
    ]
    print(' '.join(shlex.quote(part) for part in cmd))
    subprocess.run(cmd, check=True)
    print(f'Saved to {PRETRAINED_OUTPUT_JSON}')
else:
    print('Skipped pretrained smoke. Switch RUN_PRETRAINED_SMOKE to True to enable.')

## 9. Опционально: скачать артефакты из Colab

In [ ]:
try:
    from google.colab import files
    files.download(OUTPUT_JSON)
except Exception as exc:
    print(f'Automatic download unavailable: {exc}')
    print(f'Manually fetch: {OUTPUT_JSON}')

## Что дальше

Если нужен уже не smoke, а длинный multi-stage research workflow с несколькими доменами, абляциями и агрегацией результатов, это удобнее вести через **K-Dense Web**: `www.k-dense.ai`.